# 4) Probabilistic forecasts — Exercises

**Goals:** implement and test the pinball loss, check calibration at several levels, try to repair seasonal miscalibration, size a reserve from forecast errors, and compare two kinds of solar interval.

The setup cell holds the lecture's honest 2024 forecast errors `ERRORS_2024`, the 2025 point forecast `POINT_2025` and actuals `ACTUAL_2025` (MW), and the solar data `SOLAR` (MW, 1,500 MW fleet).

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, solar_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
FORMULA = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
honest = df.assign(temperature=df.temperature_forecast)

# the lecture's honest errors (2023 model forecasting 2024) and its 2025 point forecast (2023-2024 model)
ERRORS_2024 = (df.demand - smf.ols(FORMULA, df.loc["2023"]).fit().predict(honest)).loc["2024"]
POINT_2025 = smf.ols(FORMULA, df.loc[:"2024"]).fit().predict(honest).loc["2025"]
ACTUAL_2025 = df.demand.loc["2025"]
SOLAR = solar_data(df)

### Warm-ups

**1. The pinball loss**

Write `pinball(y, q, alpha)`. Then check its defining property on 100,000 draws from a normal distribution with mean 0 and standard deviation 1: over a grid of candidate values q from −3 to 3, the loss for α = 0.9 must be smallest near the true 90 % quantile, 1.2816.

In [2]:
def pinball(y, q, alpha: float) -> float:
    """Mean pinball loss of the alpha-quantile forecast q."""
    ### BEGIN SOLUTION
    d = np.asarray(y, float) - np.asarray(q, float)
    return float(np.mean(np.maximum(alpha * d, (alpha - 1) * d)))
    ### END SOLUTION

In [3]:
draws = np.random.default_rng(0).normal(0, 1, 100_000)
grid = np.round(np.arange(-3, 3.001, 0.01), 2)
losses = pd.Series([pinball(draws, q, 0.9) for q in grid], index=grid)
assert abs(losses.idxmin() - 1.2816) < 0.03, "the pinball loss is minimised by the true quantile"
assert np.isclose(pinball([10.0], [8.0], 0.9), 0.9 * 2) and np.isclose(pinball([10.0], [12.0], 0.9), 0.1 * 2)
print(f"loss minimised at q = {losses.idxmin():.2f} (true 90 % quantile 1.2816)")

loss minimised at q = 1.28 (true 90 % quantile 1.2816)


**2. Calibration at several levels**

Build central intervals at nominal levels 50 %, 80 %, 90 % and 95 % from the quantiles of `ERRORS_2024` around `POINT_2025`, and report the share of 2025 hours inside each. Write `reliability(levels)`.

In [4]:
def reliability(levels=(0.5, 0.8, 0.9, 0.95)) -> pd.DataFrame:
    """Nominal level, actual 2025 coverage and width (MW) of central intervals from the 2024 errors."""
    ### BEGIN SOLUTION
    rows = {}
    for level in levels:
        lo, hi = np.quantile(ERRORS_2024, [(1 - level) / 2, (1 + level) / 2])
        inside = (ACTUAL_2025 >= POINT_2025 + lo) & (ACTUAL_2025 <= POINT_2025 + hi)
        rows[level] = {"coverage": float(inside.mean()), "width_MW": hi - lo}
    return pd.DataFrame(rows).T.rename_axis("nominal")
    ### END SOLUTION

In [5]:
rel = reliability()
assert (np.abs(rel.coverage - rel.index) < 0.05).all(), "within 5 points of nominal at every level"
assert rel.width_MW.is_monotonic_increasing
rel.round(3)

         coverage  width_MW
nominal                    
0.50        0.493   120.405
0.80        0.806   237.536
0.90        0.915   316.642
0.95        0.960   377.806

### Core

**3. Can month-specific widths repair seasonal miscalibration?**

The lecture's interval covered between 72 % and 87 % depending on the month. Build an 80 % interval from **month-specific** quantiles of `ERRORS_2024` instead of one set for the whole year. Report the spread of monthly coverage and the mean pinball loss (of the 10 % and 90 % bounds) for both versions. Write `monthly_intervals()`.

In [6]:
def monthly_intervals() -> pd.DataFrame:
    """Overall coverage, lowest and highest monthly coverage, and mean pinball loss of two 80 % intervals."""
    ### BEGIN SOLUTION
    months = ACTUAL_2025.index.month
    q_all = np.quantile(ERRORS_2024, [0.1, 0.9])
    q_month = ERRORS_2024.groupby(ERRORS_2024.index.month).quantile([0.1, 0.9]).unstack()
    variants = {"one interval for the year": (np.full(len(months), q_all[0]), np.full(len(months), q_all[1])),
                "month-specific": (q_month.loc[months, 0.1].to_numpy(), q_month.loc[months, 0.9].to_numpy())}
    rows = {}
    for name, (lo, hi) in variants.items():
        lower, upper = POINT_2025 + lo, POINT_2025 + hi
        inside = (ACTUAL_2025 >= lower) & (ACTUAL_2025 <= upper)
        by_month = inside.groupby(months).mean()
        rows[name] = {"coverage": float(inside.mean()), "lowest_month": by_month.min(), "highest_month": by_month.max(),
                      "pinball_mean": (pinball(ACTUAL_2025, lower, 0.1) + pinball(ACTUAL_2025, upper, 0.9)) / 2}
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [7]:
monthly = monthly_intervals()
assert np.isclose(monthly.loc["one interval for the year", "coverage"], 0.806, atol=0.001)
monthly.round(3)

                           coverage  lowest_month  highest_month  pinball_mean
one interval for the year     0.806         0.721          0.870        16.700
month-specific                0.766         0.661          0.921        17.724

**4. Sizing a reserve**

A TSO wants enough upward reserve to cover the demand forecast's under-forecasts in 99 % of hours. Size it from `ERRORS_2024` (demand − forecast) as their 99 % quantile, then check 2025: in how many hours did demand exceed the forecast by more than the reserve, and by how much at worst? Repeat for 95 % and 99.9 %. Write `reserve_sizing(levels)`.

In [8]:
def reserve_sizing(levels=(0.95, 0.99, 0.999)) -> pd.DataFrame:
    """Reserve (MW) sized on 2024 errors, and the 2025 hours and worst shortfall beyond it."""
    ### BEGIN SOLUTION
    excess = ACTUAL_2025 - POINT_2025
    rows = {}
    for level in levels:
        reserve = float(np.quantile(ERRORS_2024, level))
        short = excess - reserve
        rows[level] = {"reserve_MW": reserve, "hours_short_2025": int((short > 0).sum()),
                       "expected_hours": (1 - level) * len(excess), "worst_shortfall_MW": float(short.max())}
    return pd.DataFrame(rows).T.rename_axis("level")
    ### END SOLUTION

In [9]:
reserves = reserve_sizing()
assert reserves.reserve_MW.is_monotonic_increasing
assert reserves.loc[0.99, "hours_short_2025"] > 0
reserves.round(1)

       reserve_MW  hours_short_2025  expected_hours  worst_shortfall_MW
level                                                                  
0.950       149.1             432.0           438.0               174.5
0.990       215.1              61.0            87.6               108.5
0.999       282.6               9.0             8.8                41.0

**5. Interpretation**

Did month-specific quantiles fix the seasonal miscalibration — and did the pinball loss reward them? What did the reserve check reveal about sizing from one year of errors, especially at 99.9 %?

<!-- BEGIN SOLUTION -->
No — month-specific quantiles made things **worse**. Overall coverage fell from 80.6 % to 76.6 %, the monthly spread widened from 72–87 % to 66–92 %, and the pinball loss rose from 16.70 to 17.72. Each month's quantiles now rest on about 720 hours of 2024's errors, under 2024's particular weather in that month; 2025's June is not 2024's June, so the month-specific widths are partly noise. This is overfitting, applied to the uncertainty rather than the point forecast: a more detailed interval needs more data than one year to estimate. A fix that conditions on something that really drives the uncertainty — the temperature forecast, say, or several years of errors per season — has a better chance than conditioning on the calendar month alone. The pinball loss caught the failure that a quick look at overall coverage might have missed.

The reserve check works well at 95 % (432 hours short against 438 expected) and at 99.9 % (9 against 8.8); at 99 % 2025 was a little kinder than 2024 (61 against 88). But the 99.9 % reserve is estimated from the largest **9 errors** of 2024 — 0.1 % of 8,784 hours — so it rests on almost nothing, and next year a single storm or outage outside anything seen in 2024 would breach it by far more than 41 MW. These synthetic errors are close to normal, with thin tails; real forecast errors have heavier tails (holidays, extreme weather, plant trips). Reserves sized for rare events need far more than one year of errors, or a model of the tail itself.
<!-- END SOLUTION -->

### Challenge

**6. Solar: one width or a width that follows the forecast?**

Compare two 80 % solar intervals for 2025, both built from 2024's errors and clipped to 0–1,500 MW: (a) one constant width from the quantiles of 2024's **daylight** errors, and (b) the lecture's widths by forecast level (bins −1, 1, 100, 300, 600, 900, 1,500 MW). Score both on **daylight** hours of 2025 (actual output above 1 MW): coverage, mean width and mean pinball loss. Write `solar_intervals()`.

In [10]:
def solar_intervals() -> pd.DataFrame:
    """Daylight coverage, mean width (MW) and mean pinball loss of two 80 % solar intervals for 2025."""
    ### BEGIN SOLUTION
    s24, f24 = SOLAR.solar.loc["2024"], SOLAR.solar_forecast.loc["2024"]
    s, f = SOLAR.solar.loc["2025"], SOLAR.solar_forecast.loc["2025"]
    day24, day = s24 > 1, s > 1
    q_const = np.quantile((s24 - f24)[day24], [0.1, 0.9])
    bins = [-1, 1, 100, 300, 600, 900, 1500]
    q_bin = (s24 - f24).groupby(pd.cut(f24, bins), observed=True).quantile([0.1, 0.9]).unstack()
    level = pd.cut(f, bins)
    variants = {"constant width": (f + q_const[0], f + q_const[1]),
                "width by forecast level": (f + q_bin.loc[level, 0.1].to_numpy(), f + q_bin.loc[level, 0.9].to_numpy())}
    rows = {}
    for name, (lo, hi) in variants.items():
        lo, hi = lo.clip(0, 1500)[day], hi.clip(0, 1500)[day]
        y = s[day]
        rows[name] = {"coverage_daylight": float(((y >= lo) & (y <= hi)).mean()), "mean_width_MW": float((hi - lo).mean()),
                      "pinball_mean": (pinball(y, lo, 0.1) + pinball(y, hi, 0.9)) / 2}
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [11]:
solar_table = solar_intervals()
assert solar_table.loc["width by forecast level", "pinball_mean"] < solar_table.loc["constant width", "pinball_mean"]
solar_table.round(3)

                         coverage_daylight  mean_width_MW  pinball_mean
constant width                       0.792        223.133        18.940
width by forecast level              0.778        212.980        17.194

**7. Interpretation**

Which solar interval is better, and how do coverage, width and pinball loss each tell the story? Why does the constant-width interval look acceptable on coverage alone?

<!-- BEGIN SOLUTION -->
The level-dependent interval is better, although its daylight coverage is a little **lower** (77.8 % against 79.2 %). The constant width is too wide when little output is expected and too narrow when much is: it covers 97 % of the daylight hours with a forecast below 100 MW, but only 61–62 % of those with a forecast above 600 MW. The two errors cancel in the average coverage, so it looks well calibrated. It pays for that with a wider interval (223 against 213 MW on average) placed in the wrong hours, and the pinball loss — which judges each hour's bounds against that hour's outcome — sees through it: 18.9 against 17.2. Coverage alone would have picked the wrong interval; coverage plus sharpness, or the pinball loss that combines both, picks the right one.
<!-- END SOLUTION -->